In [ ]:
cd(joinpath(@__DIR__, ".."))   # run from the repository root

In [ ]:
using VoronoiMeshes,TensorsLite, TensorsLiteGeometry, ImmutableVectors, LinearAlgebra, NCDatasets, MPASTools
using StaticArrays, SparseArrays
using CairoMakie
include("src/juliaDEC.jl")

In [ ]:
mesh = VoronoiMesh("meshes/mesh6_regular.nc")

function rk4_step(f, u, t, dt)
    k1 = f(u, t)
    k2 = f(u + 0.5*dt*k1, t + 0.5*dt)
    k3 = f(u + 0.5*dt*k2, t + 0.5*dt)
    k4 = f(u + dt*k3, t + dt)
    return u + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
end

function initial_condition1(x)
    xp = mesh.x_period
    yp = mesh.y_period
    center = [0.3, 0.5]
    sigma = 0.05

    x_vec = TensorsLite.Vec(x = x[1], y = x[2])
    center_vec = TensorsLite.Vec(x = center[1], y = center[2])
    x_raw = closest(center_vec, x_vec, xp, yp)
    y = [x_raw.x, x_raw.y]
    return exp(-(norm(y - center))^2/(2*sigma^2))
end

struct AdvectionProblem #we solve du/dt + iᵥ(du) = 0
    mesh::VoronoiMesh
    velocity::Function  # Vector field function
    rho::Float64
    initial_condition::Function
    dt::Float64
    T::Float64
end

In [ ]:
velocity = x -> [1.0, 0.0]  # Constant field
reverse_velocity = x -> [-1.0, 0.0]
velocity2 = x -> [-sin(pi*x[1])^2*sin(2*pi*x[2]), sin(pi*x[2])^2*sin(2*pi*x[1])] # Non-constant field
reverse_velocity2 = x -> [sin(pi*x[1])^2*sin(2*pi*x[2]), -sin(pi*x[2])^2*sin(2*pi*x[1])]
rho = 1.0
T1 = 0.4
T2 = 2.0
dt = 0.001

p1 = AdvectionProblem(mesh, velocity, rho, initial_condition1, dt, T1)
p1_rev = AdvectionProblem(mesh, reverse_velocity, rho, initial_condition1, dt, T1)
p2 = AdvectionProblem(mesh, velocity2, rho, initial_condition1, dt, T2)
p2_rev = AdvectionProblem(mesh, reverse_velocity2, rho, initial_condition1, dt, T2)

u_initial_primal_0form = Cochain(initial_condition1.(mesh.cells.position), 0, true)
u_initial_dual_0form = Cochain(initial_condition1.(mesh.vertices.position), 0, false)
u_initial_dual_2form = Cochain(initial_condition1.(mesh.cells.position) .* mesh.cells.area, 2, false);

println("length of mesh = ", maximum(mesh.edges.lengthDual))

In [ ]:
# Plot initial condition
fig = Figure(size=(800, 600))
ax = Axis(fig[1, 1], title = "Initial Condition")
fig, ax = plot_cochain(fig, ax, u_initial_primal_0form, mesh, :rainbow)
fig

## Caso 1 - Discretizando diretamente a equação de transporte:
$$
\frac{\partial u}{\partial t} + \text{div}(u\vec{v}) = 0
$$

Conisiderando que o campo vetorial é uma $1$-forma, reescrevemos o problema como

$$
\frac{\partial u}{\partial t} + \star d \star (u\wedge{v}) = 0.
$$

Utilizando os operadores DEC, resulta que
$$
\frac{\partial u_h}{\partial t} + \star_{2}^{\text{dual}} d_{1}^{\text{dual}}\star_{1}^{\text{primal}} W_v \,u_h  = 0,
$$
onde $W_v$ representa a discretização do produto $u \wedge v$.

### Transporte com campo constante

In [ ]:
# Initialize solution
problem = p1
u = Cochain(u_initial_primal_0form.values, 0, true)

# Precompute operators
v = discretize_vector_field_dual(problem.mesh, problem.velocity)
d1 = d1_matrix_dual(problem.mesh)
#star1 = star(problem.mesh, 1, true) 
star1 = hodge_star_lsq_matrix(problem.mesh, true, false)
star2 = star(problem.mesh, 2, false)
wedge10 = wedge_product_10_matrix(problem.mesh, v)

# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return - star2 * d1 * star1 * wedge10 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 0, true)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2,ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
maximum(abs.(u.values - u_initial_primal_0form.values))

In [ ]:
fig2

In [ ]:
fig3 = Figure()
ax3 = Axis(fig3[1, 1])
aux = Cochain(u_initial_primal_0form.values - u.values, 0, true)
fig3,ax3 = plot_cochain(fig3, ax3, aux, problem.mesh, :rainbow)

In [ ]:
fig3

### Avançando no tempo com campo reverso

In [ ]:
problem = p1_rev

# Precompute operators
v = discretize_vector_field_dual(problem.mesh, problem.velocity)
wedge10 = wedge_product_10_matrix(problem.mesh, v)

# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return - star2 * d1 * star1 * wedge10 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 0, true)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2,ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
maximum(abs.(u.values - u_initial_primal_0form.values))

In [ ]:
fig2

### Transporte com campo não constante

In [ ]:
# Define problem
problem = p2

# Initialize solution
u = Cochain(u_initial_primal_0form.values, 0, true)

# Precompute operators
v = discretize_vector_field_dual(problem.mesh, problem.velocity)
wedge10 = wedge_product_10_matrix(problem.mesh, v)

# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return - star2 * d1 * star1 * wedge10 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 0, true)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2,ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
println("Erro na norma Infinito:")
maximum(abs.(u.values - u_initial_primal_0form.values))


In [ ]:
fig2

#### Utilizando u encontrado como condição inicial e "voltando para trás."

In [ ]:
problem = p2_rev

# Precompute operators
v = discretize_vector_field_dual(problem.mesh, problem.velocity)
wedge10 = wedge_product_10_matrix(problem.mesh, v)

# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return - star2 * d1 * star1 * wedge10 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 0, true)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2,ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
println("Erro na norma Infinito:")
println(maximum(abs.(u.values - u_initial_primal_0form.values)))

In [ ]:
fig3 = Figure()
ax3 = Axis(fig3[1, 1])
aux = Cochain(u_initial_primal_0form.values - u.values, 0, true)
fig3,ax3 = plot_cochain(fig3, ax3, aux, problem.mesh, :rainbow)

In [ ]:
fig3

### Caso 2 - Discretizando o problema de transporte de Lie com 2-forma dual:
$$
\frac{\partial u_h}{\partial t} + \mathcal{L}_v u = 0,
$$
onde $ \mathcal{L}_v u = i_v(du) + d i_v (u)$. Tome $u_h$ uma $2$-forma discreta dual. Então, resolvemos o problema
$$
\frac{\partial u_h}{\partial t} + (-1)^{k(2-k)} d \star (\star u \wedge v) = 0.
$$
Dessa forma, temos que
$$
\frac{\partial u_h}{\partial t} + d_1^{dual} \star_1^{primal} W_v \star_2^{dual} u_h  = 0.
$$


### Transporte com campo constante

In [ ]:
problem = p1
# Initialize solution
u = Cochain(u_initial_dual_2form.values, 2, false)

v = discretize_vector_field_dual(problem.mesh, problem.velocity)
d1 = d1_matrix_dual(problem.mesh)
#star1 = star(problem.mesh, 1, true)
star1 = hodge_star_lsq_matrix(problem.mesh, true, false)
star2 = star(problem.mesh, 2, false)
wedge10 = wedge_product_10_matrix(problem.mesh, v)
# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return  -d1 * star1 * wedge10 * star2 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 2, false)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2, ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow)
println("Erro na norma Infinito:")
maximum(abs.(u.values - u_initial_dual_2form.values))

In [ ]:
fig2

In [ ]:
fig3 = Figure()
ax3 = Axis(fig3[1, 1])
aux = Cochain(u_initial_dual_2form.values - u.values, 2, false)
fig3,ax3 = plot_cochain(fig3, ax3, aux, problem.mesh, :rainbow)

In [ ]:
fig3

### Teste com campo não constante

In [ ]:
problem = p2

# Initialize solution
u = Cochain(u_initial_dual_2form.values, 2, false)
v = discretize_vector_field_dual(problem.mesh, problem.velocity)

d1 = d1_matrix_dual(problem.mesh)
#star1 = star(problem.mesh, 1, true)
star1 = hodge_star_lsq_matrix(problem.mesh, true, false)
star2 = star(problem.mesh, 2, false)
wedge10 = wedge_product_10_matrix(problem.mesh, v)
# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return  -d1 * star1 * wedge10 * star2 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 2, false)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2, ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);

In [ ]:
fig2

### Voltando para trás com campo não constante

In [ ]:
problem = p2_rev

v = discretize_vector_field_dual(problem.mesh, problem.velocity)
wedge10 = wedge_product_10_matrix(problem.mesh, v)
# Define the right-hand side function for RK4
function rhs(u_values, t)
    # Compute -iᵥ(du) = (-1)^(k(2-k)) *(*du ∧ v)
    return  -d1 * star1 * wedge10 * star2 * u_values
end
t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(rhs, u.values, t, dt), 2, false)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2, ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
println("Erro na norma Infinito:")
maximum(abs.(u.values - u_initial_dual_2form.values))

In [ ]:
fig2

In [ ]:
fig3 = Figure()
ax3 = Axis(fig3[1, 1])
aux = Cochain(u_initial_dual_2form.values - u.values, 2, false)
fig3,ax3 = plot_cochain(fig3, ax3, aux, problem.mesh, :rainbow)

In [ ]:
fig3

### Caso 3 - Discretizando o problema de transporte de Lie com 0-forma dual:
$$
\frac{\partial u_h}{\partial t} + \mathcal{L}_v u = 0,
$$
onde $ \mathcal{L}_v u = i_v(du) + d i_v (u)$. Tome $u_h$ uma $0$-forma discreta dual. Então, resolvemos o problema
$$
\frac{\partial u_h}{\partial t} + (-1)^{k(2-k)} \star (\star d u \wedge v) = 0.
$$
Dessa forma, temos que
$$
\frac{\partial u_h}{\partial t} - \star_2^{primal} W_v \star_1^{\text{dual}} d_0^{\text{dual}} u_h  = 0.
$$


### Transporte com campo constante

In [ ]:
problem = p1

# Initialize solution
u = Cochain(u_initial_dual_0form.values, 0, false)
v = discretize_vector_field_dual(problem.mesh, problem.velocity)
d0 = d0_matrix_dual(problem.mesh)
star2 = star(problem.mesh, 2, true)
#star1 = star(problem.mesh, 1, false)
star1 = hodge_star_lsq_matrix(problem.mesh, false, false)
wedge11 = wedge_product_11_matrix(problem.mesh, v)

# iᵥ(du)
f(u_values, t) = star2 * wedge11 * star1 * d0 * u_values

t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(f, u.values, t, dt), 0, false)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2, ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
println("Erro na norma Infinito:")
maximum(abs.(u.values - u_initial_dual_0form.values))

In [ ]:
fig2

In [ ]:
fig3 = Figure()
ax3 = Axis(fig3[1, 1])
aux = Cochain(u_initial_dual_0form.values - u.values, 0, false)
fig3,ax3 = plot_cochain(fig3, ax3, aux, problem.mesh, :rainbow)

In [ ]:
fig3

### Teste com campo não constante

In [ ]:
problem = p2

# Initialize solution
u = Cochain(u_initial_dual_0form.values, 0, false)
v = discretize_vector_field_dual(problem.mesh, problem.velocity)
d0 = d0_matrix_dual(problem.mesh)
star2 = star(problem.mesh, 2, true)
#star1 = star(problem.mesh, 1, false)
star1 = hodge_star_lsq_matrix(problem.mesh, false, false)
wedge11 = wedge_product_11_matrix(problem.mesh, v)

# iᵥ(du)
f(u_values, t) = star2 * wedge11 * star1 * d0 * u_values

t = 0.0
j = 0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(f, u.values, t, dt), 0, false)
    t += dt
    j += 1
    if j % 100 == 0
        println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
    end
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2, ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
println("Erro na norma Infinito:")
maximum(abs.(u.values - u_initial_dual_0form.values))

In [ ]:
fig2

### Voltando para trás com campo não constante

In [ ]:
problem = p2_rev

v = discretize_vector_field_dual(problem.mesh, problem.velocity)
wedge11 = wedge_product_11_matrix(problem.mesh, v)

# iᵥ(du)
f(u_values, t) = star2 * wedge11 * star1 * d0 * u_values

t = 0.0
while t < problem.T
    dt = min(problem.dt, problem.T - t)
    u = Cochain(rk4_step(f, u.values, t, dt), 0, false)
    t += dt
    println("t = $t, max value = $(maximum(u.values)),   min value = $(minimum(u.values))")
end

fig2 = Figure(size=(800, 600))
ax2 = Axis(fig2[1, 1], title = "Solution after $T seconds")
fig2, ax2 = plot_cochain(fig2, ax2, u, problem.mesh, :rainbow);
println("Erro na norma Infinito:")
maximum(abs.(u.values - u_initial_dual_0form.values))

In [ ]:
fig2

In [ ]:
fig3 = Figure()
ax3 = Axis(fig3[1, 1])
aux = Cochain(u_initial_dual_0form.values - u.values, 0, false)
fig3,ax3 = plot_cochain(fig3, ax3, aux, problem.mesh, :rainbow)

In [ ]:
fig3